In [2]:
from datasets import load_dataset
dataset = list(load_dataset("lmarena-ai/arena-human-preference-100k")["train"])

README.md:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

c:\Users\plaban\AppData\Local\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\plaban\.cache\huggingface\hub\datasets--lmarena-ai--arena-human-preference-100k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


arena-explorer-preference-100k.parquet:   0%|          | 0.00/380M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/106134 [00:00<?, ? examples/s]

In [3]:
print(f"Total number of samples: {len(dataset)}")
dataset_english = [d for d in dataset if d["language"] == "English"]
print(f"Number of English samples: {len(dataset_english)}")

Total number of samples: 106134
Number of English samples: 57675


In [ ]:
from matplotlib import pyplot as plt
from collections import Counter
import json

min_length = 100
max_length = 2000

dataset_en_creative = [d for d in dataset_english if d["category_tag"]["criteria_v0.1"]["creativity"] and d["turn"] == 1]
print(f"Number of creative samples: {len(dataset_en_creative)}")
# dataset_en_creative[0]

print(Counter(d['winner'] for d in dataset_en_creative))

dataset_non_ties = [d for d in dataset_en_creative if d['winner'] in ["model_a", "model_b"]]
print(f"Number of non-ties: {len(dataset_non_ties)}")

dataset_selected = [d for d in dataset_non_ties if min_length <= d['conversation_a'][-1]['num_tokens'] < max_length and min_length <= d['conversation_b'][-1]['num_tokens'] < max_length]
print(f"Number of selected samples: {len(dataset_selected)}")

with open("data/lmarena_initial_filters.json", "w") as f:
    json.dump(dataset_selected, f)

Number of creative samples: 15198
Counter({'model_b': 5156, 'model_a': 4852, 'tie': 2719, 'tie (bothbad)': 2471})
Number of non-ties: 10008
Number of selected samples: 7981


In [15]:
with open("data/lmarena_initial_filters.json", "r") as f:
    dataset_short_length = json.load(f)

with open("prompts/pairwise_pref.txt", "r") as f:
    pairwise_prompt = f.read()

verified_dataset = [d for d in dataset_short_length if "creative_cls_gpt4o" in d and d['creative_cls_gpt4o']['is_creative_writing_task'] == "yes"]

final_dataset = []
for didx, d in enumerate(verified_dataset):
    if didx % 2 == 0:
        sample1 = {"id": f"test-lmarena-{len(final_dataset)}", "original_id": d['question_id'], "split": "test", "sample_type": "pairwise-lmarena", "model_a": d['model_a'], "model_b": d['model_b']}
        para1, para2 = d['conversation_a'][1]["content"], d['conversation_b'][1]["content"]
        sample1["user_instruction"] = d['conversation_a'][0]["content"]
        sample1["text_input"] = pairwise_prompt.replace("[[PARAGRAPH1]]", para1).replace("[[PARAGRAPH2]]", para2)
        sample1["paragraph1"] = para1
        sample1["paragraph2"] = para2
        sample1["reference_preference"] = "1" if d['winner'] == "model_a" else "2"
        final_dataset.append(sample1)
    else:
        sample2 = {"id": f"test-lmarena-{len(final_dataset)}", "original_id": d['question_id'], "split": "test", "sample_type": "pairwise-lmarena", "model_a": d['model_b'], "model_b": d['model_a']}
        para1, para2 = d['conversation_b'][1]["content"], d['conversation_a'][1]["content"]
        sample2["user_instruction"] = d['conversation_b'][0]["content"]
        sample2["text_input"] = pairwise_prompt.replace("[[PARAGRAPH1]]", para1).replace("[[PARAGRAPH2]]", para2)
        sample2["paragraph1"] = para1
        sample2["paragraph2"] = para2
        sample2["reference_preference"] = "2" if d['winner'] == "model_a" else "1"
        final_dataset.append(sample2)

print(len(final_dataset))
with open("data/lamp_PRGSH_test.json", "r") as f:
    lamp_PRGSH_test = json.load(f)

lamp_PRGSH_test = [d for d in lamp_PRGSH_test if d["sample_type"] != "pairwise-lmarena"] # remove what was previously added
lamp_PRGSH_test += final_dataset

with open("data/lamp_PRGSH_test.json", "w") as f:
    json.dump(lamp_PRGSH_test, f, indent=4)

print(Counter(d['sample_type'] for d in lamp_PRGSH_test))

1959
Counter({'pairwise-lmarena': 1959, 'pairwise-gold': 1206, 'pairwise-silver': 1120, 'reward': 430, 'pairwise': 404, 'pairwise-h': 300, 'pairwise-P1': 215, 'pairwise-P2': 215, 'pairwise-P3': 209, 'pairwise-P4': 199, 'pairwise-P5': 183, 'pairwise-P6': 159, 'pairwise-art': 144, 'pairwise-P7': 138})
